In [29]:
import numpy as np
import torch
from transformers import AutoTokenizer

from megatron.core.tokenizers import MegatronTokenizer
from megatron.bridge.data.datasets.utils import _chat_preprocess

In [25]:
tokenizer = MegatronTokenizer.from_pretrained("/EU-Model-Builder-SAs/user_homes/osudakov/nemotron-3-super-120b")

In [26]:
CHAT_SAMPLE = [{'role': 'system',
  'content': 'This is a system prompt.'},
 {'role': 'user',
  'content': "This is a user question"},
  {'role': 'assistant',
  'content': "This is assistant answer.",
  'reasoning_content': ''}
]

preprocessed = _chat_preprocess(CHAT_SAMPLE, tokenizer)
print(preprocessed)

{'input_ids': tensor([   10, 25708,  1010,  4380,  1395,  1261,  2663, 16925,  1046,    11,
         1010,    10,  3263,  1010,  4380,  1395,  1261,  3330,  4098,    11,
         1010,    10,  1503, 19464,  1010,    12,    13,  4380,  1395, 27089,
         4832,  1046,    11,  1010]), 'loss_mask': tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True, False, False, False,
        False, False, False, False]), 'context_ids': tensor([   10, 25708,  1010,  4380,  1395,  1261,  2663, 16925,  1046,    11,
         1010,    10,  3263,  1010,  4380,  1395,  1261,  3330,  4098,    11,
         1010,    10,  1503, 19464,  1010,    12,    13,  4380,  1395, 27089,
         4832,  1046,    11,  1010]), 'answer_ids': tensor([], dtype=torch.int64)}


In [27]:
tokenizer._tokenizer.ids_to_text(preprocessed["input_ids"][preprocessed["loss_mask"]].numpy())

'<|im_start|>system\nThis is a system prompt.<|im_end|>\n<|im_start|>user\nThis is a user question<|im_end|>\n<|im_start|>assistant\n<think></think>'

In [32]:
tokenizer._tokenizer.ids_to_text(preprocessed["input_ids"][torch.logical_not(preprocessed["loss_mask"])].numpy())

'This is assistant answer.<|im_end|>\n'

In [34]:
TOOL_CALLING_SAMPLE = [{'role': 'system',
  'content': 'This is a system prompt.'},
 {'role': 'user',
  'content': "This is a user question"},
 {'role': 'assistant',
  'tool_calls': [{'type': 'function',
    'id': 'call_f174a28a646b4b2787328057',
    'function': {'name': 'test_tool',
     'arguments': '{"argument": "This is a test argument."}'}}],
  'reasoning_content': 'This is reasoning content.',
  'content': ''},
 {'role': 'tool',
  'tool_call_id': 'call_f174a28a646b4b2787328057',
  'content': '\n\n{"confirmation": "This is tool content."}'},
 {'role': 'assistant',
  'content': "This is assistant answer.",
  'reasoning_content': ''}]


TOOLS = [
 {'type': 'function',
  'function': {'name': 'test_tool',
   'description': 'Tool for a test.',
   'parameters': {'properties': {'argument': {'type': 'string'}},
    'required': ['argument']},
   'strict': True}},
  ]

preprocessed = _chat_preprocess(TOOL_CALLING_SAMPLE, tokenizer, TOOLS)
print(preprocessed)

{'input_ids': tensor([    10,  25708,   1010,   4380,   1395,   1261,   2663,  16925,   1338,
          1035,  49977,   1267,   4568,   1736,   4731,   1317,   1278,   3629,
          7389,   2100,   1060,  47742,   1561,   1060,   5165,   1561,   1060,
          2391,   1062,   4417,   7198,   1336,   1885,   2391,   1561,   1060,
         14653,   1062,  22483,   1394,   1261,   2688,  15342,  14653,   1561,
          1060,  26204,   1561,   1060,  31960,   1561,   1060,   2391,   1062,
         69992,   1885,   2391,   1561,   1060,   4994,   1062,   3607,   1885,
          4994,   1561,   1885,  31960,   1561,   1060,  15760,   1062,   4651,
         69992,   4964,   1885,  15760,   1561,   1885,  26204,   1561,   1060,
        121182,   1062,   7848,   1885, 121182,   1561,   1885,   5165,   1561,
          1885,  47742,   3318,   5475,   1636,  12729,   1317,   3690,   1261,
          2254, 101803,  22695,   1294,   1278,   3629,   8174,   1454,  14089,
         39398,   2100,   

In [35]:
tokenizer._tokenizer.ids_to_text(preprocessed["input_ids"][preprocessed["loss_mask"]].numpy())

'<|im_start|>system\nThis is a system prompt.\n\n# Tools\n\nYou have access to the following functions:\n\n<tools>\n<function>\n<name>test_tool</name>\n<description>Tool for a test.</description>\n<parameters>\n<parameter>\n<name>argument</name>\n<type>string</type>\n</parameter>\n<required>["argument"]</required>\n</parameters>\n<strict>True</strict>\n</function>\n</tools>\n\nIf you choose to call a function ONLY reply in the following format with NO suffix:\n\n<tool_call>\n<function=example_function_name>\n<parameter=example_parameter_1>\nvalue_1\n</parameter>\n<parameter=example_parameter_2>\nThis is the value for the second parameter\nthat can span\nmultiple lines\n</parameter>\n</function>\n</tool_call>\n\n<IMPORTANT>\nReminder:\n- Function calls MUST follow the specified format: an inner <function=...></function> block must be nested within <tool_call></tool_call> XML tags\n- Required parameters MUST be specified\n- You may provide optional reasoning for your function call in nat

In [36]:
tokenizer._tokenizer.ids_to_text(preprocessed["input_ids"][torch.logical_not(preprocessed["loss_mask"])].numpy())

'\nThis is reasoning content.\n</think>\n<tool_call>\n<function=test_tool>\n<parameter=argument>\nThis is a test argument.\n</parameter>\n</function>\n</tool_call>\n<|im_end|>\nThis is assistant answer.<|im_end|>\n'